# Riduzione della dimensionalità con PCA

L'analisi delle componenti principali (PCA, *Principal Component Analysis*) è una tecnica di riduzione della dimensionalità delle caratteristiche. Non si basa sulla ricerca delle caratteristiche più importanti, come accade con tecniche basate su modelli di tipo feature selection (ad esempio le foreste casuali), ma trasforma il dataset — composto da variabili correlate e quindi ridondanti — proiettandolo in un nuovo spazio, di dimensionalità inferiore, in cui le variabili risultano non correlate.

![./05_01.png](./21_01.png)

I dati in figura risultano correlati: al crescere della `x` cresce la `y`. La PCA cerca la direzione in cui c'è massima varianza dei dati (i punti risultano più sparpagliati), nel caso della figura la direzione `PC1`. Dopo di che, se cerca l'asse perpendicolare a `PC1` con varianza massima residua e si continua in questo modo. Questi assi sono tutti ortogonali tra di loro, quindi i punti proiettati su questi sono indipendenti.

La compressione di dimensionalità, trova i `k < d` assi con varianza massima e proietta i dati nel nuovo spazio `k`-dimensionale.
## Come funziona

Sia $X$ una matrice in $n\times d$

1. Centrare $X$ in modo che la media sia 0.
2. Calcolare la matrice di covarianza $C$ di $X^T$:
$$
C = \frac{1}{n-1}X^T X
$$
$C$ avrà dimensione $d\times d$.

1. Trovare gli autovalori e gli autovettori di $C$: gli autovettori sono le direzioni principali. mentre gli autovalori indicano quanta varianza viene spiegata da ciascuna di esse. La somma degli autovalori indica la varianza totale.
2. Selezionare i $k$ autovettori che corrispondono agli autovalori massimi, ottenendo la matrice $W$ in $d\times k$.
3. Proiettare $X$ su $W$, definendo il nuovo insieme dei dati come
$$
X_{PCA} = X\times W
$$
che avrà dimension $n\times k$

Ora il dataset $X_{PCA}$ può essere utilizzato al posto di $X$ per eseguire, ad esempio, operazioni di classificazione.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=10000,        # numero di campioni
    n_features=100,          # due feature per visualizzare facilmente
    n_informative=10,       # entrambe informative
    n_redundant=0,         # nessuna feature ridondante
    n_repeated=0,          # nessuna feature duplicata
    n_classes=2,           # classificazione binaria
    n_clusters_per_class=1,# un solo cluster per classe
    class_sep=1.0,         # maggiore separazione tra le classi
    flip_y=0.1,              # nessun rumore nelle etichette
    random_state=30,
)


In [124]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

n = X_train.shape[0]

In [125]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = np.mean(y_test == y_pred)

print(accuracy)

0.8703333333333333


Applicazione PCA

In [ ]:
# matrice di covarianza
X_train_c = X_train - X_train.mean(axis=0)

C = (X_train_c.T @ X_train_c) / (n - 1)


$$
C =
\frac{1}{n-1}
\begin{bmatrix}
\sum_{i=1}^n (x_{i1}-\mu_1)(x_{i1}-\mu_1) & \sum_{i=1}^n (x_{i1}-\mu_1)(x_{i2}-\mu_2) & \cdots & \sum_{i=1}^n (x_{i1}-\mu_1)(x_{id}-\mu_d) \\
\sum_{i=1}^n (x_{i2}-\mu_2)(x_{i1}-\mu_1) & \sum_{i=1}^n (x_{i2}-\mu_2)(x_{i2}-\mu_2) & \cdots & \sum_{i=1}^n (x_{i2}-\mu_2)(x_{id}-\mu_d) \\
\vdots & \vdots & \ddots & \vdots \\
\sum_{i=1}^n (x_{id}-\mu_d)(x_{i1}-\mu_1) & \sum_{i=1}^n (x_{id}-\mu_d)(x_{i2}-\mu_2) & \cdots & \sum_{i=1}^n (x_{id}-\mu_d)(x_{id}-\mu_d)
\end{bmatrix}
$$


$$
C =
\begin{bmatrix}
\mathrm{Var}(x_1) & \mathrm{Cov}(x_1, x_2) & \cdots & \mathrm{Cov}(x_1, x_d) \\
\mathrm{Cov}(x_2, x_1) & \mathrm{Var}(x_2) & \cdots & \mathrm{Cov}(x_2, x_d) \\
\vdots & \vdots & \ddots & \vdots \\
\mathrm{Cov}(x_d, x_1) & \mathrm{Cov}(x_d, x_2) & \cdots & \mathrm{Var}(x_d)
\end{bmatrix}
$$

Matrice simmetrica.

Sia $v$ una direzione nello spazio delle features, ovvero di dimensione $d$ tale che $||v|| = 1$,

$$
X\cdot v
$$

È la proiezione di $X$ su $v$.

Trovare $v$ che massimizzi la varianza della proiezione di $X$ sull'asse $v$.

È noto che

$$
\mathrm{Var}(X\cdot v) = v^T  C  v
$$

Quindi

$$
\max_{v \in R^d, ||v|| = 1} (v^T C  v)
$$


### Dimostrazione che 

$$
\mathrm{Var}(X\cdot v) = v^T  C  v
$$


Assumiamo:
- $X \in \mathbb{R}^{n \times d}$ centrata (media zero per colonna)
- $v \in \mathbb{R}^{d}$
- $C = \frac{1}{n-1} X^T X$


$$
z = Xv
$$

dove $z \in \mathbb{R}^{n}$ e ogni elemento è:

$$
z_i = x_i^T v
$$

Poiché \(z\) è centrato:

$$
\mathrm{Var}(z) = \frac{1}{n-1} z^T z
$$


$$
\mathrm{Var}(Xv) = \frac{1}{n-1} (Xv)^T (Xv)
$$


$$
(Xv)^T = v^T X^T
$$

quindi:

$$
(Xv)^T (Xv) = v^T X^T X v
$$


$$
\mathrm{Var}(Xv)
= \frac{1}{n-1} v^T X^T X v
$$


$$
C = \frac{1}{n-1} X^T X
$$



$$
\mathrm{Var}(Xv) = v^T C v
$$

In [131]:
eigenvalues , eigenvectors = np.linalg.eigh(C)   # autovettori sulle colonnne

idx = np.argsort(eigenvalues)[::-1]

W = eigenvectors[:, idx[:20]]

X_train_pca = X_train_c @ W

$$
\max_{v \in R^d, ||v|| = 1} (v^T C  v)
$$

La soluzione è l'autovettore di $C$ con il massimo autovalore.

Osservazione: se $w$ è un autovettore con autovalore $\lambda$

$$
C w = \lambda w
$$

Allora

$$w^T C w = w^T\lambda w = \lambda(w^T w)$$

Ma $||w|| = 1$, allora

$$w^T C w = \lambda$$

Quindi l'autovalore è la varianza lungo la direzione dell'autovettore. 

Ogni autovettore è perpendicolare e ognuno spiega una parte dalla varianza. Partendo da quello con varianza massima si costruisce una base ortogonale di direzioni ordinate per importanza.

Il primo autovettore (con autovalore massimo) spiega la massima varianza, quello successivo massimizza la varianza residua e così via. 

### Applicazione alla classificazione

In [132]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train_pca, y_train)

X_test_pca = (X_test - X_train.mean(axis=0)) @ W

y_pred = model.predict(X_test_pca)

accuracy = np.mean(y_test == y_pred)

print(accuracy)

0.8476666666666667


In [135]:
for x in sorted(eigenvalues, reverse=True):
    print(x)

print(sum(eigenvalues))

7.112507339934971
6.958657791014022
5.187673233889804
4.665342474801909
3.2527073908087996
2.5915127328016485
2.2094500430974144
2.0815385853746546
1.464002726855619
1.2404222427008285
1.237425900878454
1.2129882187156216
1.1950708218879265
1.1874965199021579
1.1824855610690639
1.1735856908809663
1.1655733564081174
1.1599817020902403
1.155884485519769
1.1517600623337025
1.149756902210728
1.1449189903764168
1.136405356095912
1.1277505204371816
1.1215994290194597
1.1176062109489826
1.1110548616425207
1.1062194991238614
1.10222822410319
1.0995034564789898
1.0950445120467511
1.08967153172352
1.0867785399070131
1.0798126259964795
1.0752063727982797
1.0712039951449976
1.0661823554394168
1.0643718793968127
1.0578728714476464
1.0540333798595707
1.0461883758398691
1.0401022311036425
1.0381532345916689
1.0351911595959054
1.0336172371086385
1.026036131029475
1.0223606483768626
1.0209334075057876
1.0198680612684798
1.011648477945297
1.0098451059196643
1.0033358179437195
0.9963019009876012
0.991215